In [1]:
import boto3
import pandas as pd
import json
from io import BytesIO
import sagemaker
from sagemaker import image_uris

print(sagemaker.__version__)

sagemaker.config INFO - Fetched defaults config from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix
2.257.5


In [2]:
s3 = boto3.client('s3')
bucket = 'enterprise-shipment-pipeline-024532670007-ca-central-1-an'
prefix = 'load/'

response = s3.list_objects_v2(Bucket=bucket, Prefix=prefix)
part_files = [obj['Key'] for obj in response.get('Contents', []) if obj['Key'].endswith('.parquet')]

dfs = []
for key in part_files:
    obj = s3.get_object(Bucket=bucket, Key=key)
    dfs.append(pd.read_parquet(BytesIO(obj['Body'].read())))

final_df = pd.concat(dfs, ignore_index=True)
final_df.shape

(1069989, 20)

In [3]:
final_df['booking_timestamp'] = pd.to_datetime(final_df['booking_timestamp'])
final_df['month'] = final_df['booking_timestamp'].dt.to_period('M')

gold_monthly = final_df.groupby(['month', 'service_tier']).agg(
    total_credit_liability=('credit_liability', 'sum')
).reset_index()

gold_monthly.groupby('service_tier')['total_credit_liability'].sum()

service_tier
EXPRESS     482925.61
GROUND      248321.60
SAME_DAY    527049.28
Name: total_credit_liability, dtype: float64

## Baseline

In [14]:
all_months = pd.period_range(gold_monthly['month'].min(), gold_monthly['month'].max(), freq='M')
tiers = list(gold_monthly['service_tier'].unique())

series_list = []
for tier in tiers:
    tier_data = gold_monthly[gold_monthly['service_tier'] == tier].set_index('month')
    tier_data = tier_data.reindex(all_months, fill_value=0.0)
    series_list.append({
        "start": str(all_months[0].start_time.date()),
        "target": tier_data['total_credit_liability'].tolist()
    })

with open('deepar_train.json', 'w') as f:
    for series in series_list:
        f.write(json.dumps(series) + '\n')

with open('deepar_test.json', 'w') as f:
    for series in series_list:
        f.write(json.dumps(series) + '\n')  # full history

In [5]:
s3.upload_file('deepar_train.json', bucket, 'sagemaker/deepar/train/train.json')
s3.upload_file('deepar_test.json', bucket, 'sagemaker/deepar/test/test.json')

In [10]:
import boto3
import time

sm_client = boto3.client('sagemaker')

job_name = f"deepar-shipment-pipeline-{int(time.time())}"

sm_client.create_training_job(
    TrainingJobName=job_name,
    AlgorithmSpecification={
        'TrainingImage': container,
        'TrainingInputMode': 'File',
    },
    RoleArn=role,
    InputDataConfig=[
        {
            'ChannelName': 'train',
            'DataSource': {
                'S3DataSource': {
                    'S3DataType': 'S3Prefix',
                    'S3Uri': f's3://{bucket}/sagemaker/deepar/train/',
                    'S3DataDistributionType': 'FullyReplicated',
                }
            },
        },
        {
            'ChannelName': 'test',
            'DataSource': {
                'S3DataSource': {
                    'S3DataType': 'S3Prefix',
                    'S3Uri': f's3://{bucket}/sagemaker/deepar/test/',
                    'S3DataDistributionType': 'FullyReplicated',
                }
            },
        },
    ],
    OutputDataConfig={'S3OutputPath': f's3://{bucket}/sagemaker/deepar/output/'},
    ResourceConfig={
        'InstanceType': 'ml.c5.xlarge',
        'InstanceCount': 1,
        'VolumeSizeInGB': 10,
    },
    StoppingCondition={'MaxRuntimeInSeconds': 3600},
    HyperParameters={
        'time_freq': 'M',
        'prediction_length': '12',
        'context_length': '12',
        'epochs': '100',
    },
)

print(f"Training job started: {job_name}")

Training job started: deepar-shipment-pipeline-1784914294


In [12]:
import time

status = sm_client.describe_training_job(TrainingJobName=job_name)['TrainingJobStatus']
while status == 'InProgress':
    time.sleep(30)
    status = sm_client.describe_training_job(TrainingJobName=job_name)['TrainingJobStatus']
    print(status)

print(f"Final status: {status}")

InProgress
InProgress
InProgress
InProgress
InProgress
InProgress
InProgress
InProgress
InProgress
Completed
Final status: Completed


In [13]:
result = sm_client.describe_training_job(TrainingJobName=job_name)
metrics = result.get('FinalMetricDataList', [])
for m in metrics:
    print(m['MetricName'], m['Value'])

test:mean_wQuantileLoss 0.08920911699533463
train:progress 100.0
train:loss:batch 6.624838829040527
train:throughput 1263.6248779296875
test:RMSE 724.9981689453125
train:final_loss 6.594766139984131
train:loss 6.594766139984131


## Category-based

In [15]:
tiers = list(gold_monthly['service_tier'].unique())

series_list_cat = []
for i, tier in enumerate(tiers):
    tier_data = gold_monthly[gold_monthly['service_tier'] == tier].set_index('month')
    tier_data = tier_data.reindex(all_months, fill_value=0.0)
    series_list_cat.append({
        "start": str(all_months[0].start_time.date()),
        "target": tier_data['total_credit_liability'].tolist(),
        "cat": [i]
    })

with open('deepar_train_cat.json', 'w') as f:
    for series in series_list_cat:
        f.write(json.dumps(series) + '\n')

with open('deepar_test_cat.json', 'w') as f:
    for series in series_list_cat:
        f.write(json.dumps(series) + '\n')

In [16]:
s3.upload_file('deepar_train_cat.json', bucket, 'sagemaker/deepar/train_cat/train.json')
s3.upload_file('deepar_test_cat.json', bucket, 'sagemaker/deepar/test_cat/test.json')

In [17]:
job_name_cat = f"deepar-shipment-pipeline-cat-{int(time.time())}"

sm_client.create_training_job(
    TrainingJobName=job_name_cat,
    AlgorithmSpecification={'TrainingImage': container, 'TrainingInputMode': 'File'},
    RoleArn=role,
    InputDataConfig=[
        {'ChannelName': 'train', 'DataSource': {'S3DataSource': {'S3DataType': 'S3Prefix', 'S3Uri': f's3://{bucket}/sagemaker/deepar/train_cat/', 'S3DataDistributionType': 'FullyReplicated'}}},
        {'ChannelName': 'test', 'DataSource': {'S3DataSource': {'S3DataType': 'S3Prefix', 'S3Uri': f's3://{bucket}/sagemaker/deepar/test_cat/', 'S3DataDistributionType': 'FullyReplicated'}}},
    ],
    OutputDataConfig={'S3OutputPath': f's3://{bucket}/sagemaker/deepar/output_cat/'},
    ResourceConfig={'InstanceType': 'ml.c5.xlarge', 'InstanceCount': 1, 'VolumeSizeInGB': 10},
    StoppingCondition={'MaxRuntimeInSeconds': 3600},
    HyperParameters={
        'time_freq': 'M',
        'prediction_length': '12',
        'context_length': '12',
        'epochs': '100',
        'num_dynamic_feat': 'auto',
    },
)
print(f"Training job started: {job_name_cat}")

Training job started: deepar-shipment-pipeline-cat-1784915977


In [18]:
status = sm_client.describe_training_job(TrainingJobName=job_name_cat)['TrainingJobStatus']
while status == 'InProgress':
    time.sleep(30)
    status = sm_client.describe_training_job(TrainingJobName=job_name_cat)['TrainingJobStatus']
    print(status)

InProgress
InProgress
InProgress
InProgress
InProgress
InProgress
InProgress
InProgress
InProgress
Completed


In [19]:
result_cat = sm_client.describe_training_job(TrainingJobName=job_name_cat)
metrics_cat = result_cat.get('FinalMetricDataList', [])
for m in metrics_cat:
    print(m['MetricName'], m['Value'])

test:mean_wQuantileLoss 0.09481656551361084
train:progress 100.0
train:loss:batch 6.642465591430664
train:throughput 1245.037109375
test:RMSE 773.04541015625
train:final_loss 6.625233173370361
train:loss 6.625233173370361


## Method comparison

## DeepAR vs. Classical Models — Comparison

Two DeepAR training runs were evaluated: a baseline using only the raw monthly 
`credit_liability` series per tier, and a second run adding a `cat` (categorical) 
feature to distinguish GROUND/EXPRESS/SAME_DAY as related-but-distinct series.

| Model                            | Metric              | Value     |
|-----------------------------------|----------------------|-----------|
| DeepAR (no cat)                   | mean_wQuantileLoss   | `0.0892`    |
| DeepAR (no cat)                   | RMSE                 | `$724.99`   |
| DeepAR (with cat)                 | mean_wQuantileLoss   | `0.0948`    |
| DeepAR (with cat)                 | RMSE                 | `$773.05`   |
| Trend-Only Regression (classical) | MAPE                 | `7.37%`     |

**Finding 1 - cat feature:** adding `cat` made both DeepAR metrics worse (wQuantileLoss 
+6.3%, RMSE +6.6%), not better. With only 3 series and a relatively short monthly 
history each, the added categorical embedding likely introduced more parameters 
than the data could support learning from. `cat` tends to help more with larger 
sets of related series where category-level patterns have more examples to learn 
from. The no-`cat` baseline was kept as the better DeepAR candidate.

**Finding 2 - DeepAR vs. classical:** DeepAR's best result (0.0892 wQuantileLoss, 
~8.9%) did not beat Trend-Only Regression's 7.37% MAPE. With the caveat that these 
are not identical metrics (wQuantileLoss is a weighted quantile loss across DeepAR's 
full probabilistic forecast, MAPE is a simple point-estimate percentage error), 
DeepAR's error is directionally higher, not lower.

**Conclusion:** Trend-Only Regression remains the recommended production model. 
It's both simpler to deploy (no SageMaker infrastructure, training jobs, or endpoint 
management) and performed at least as well as DeepAR on this dataset. DeepAR's real 
advantage - probabilistic forecasts with confidence intervals, and its ability to 
learn shared patterns across many related series - didn't materialize here because 
the dataset only has 3 series with a relatively short, fairly clean trend. DeepAR 
would likely be more competitive on a dataset with more series (for example per-region or 
per-customer-type breakdowns) or messier, less trend-dominated data. It was evaluated 
as a production candidate but is not recommended for deployment at this data scale.